In [1]:
import groq
import google.generativeai as genai
from dotenv import load_dotenv
import os

load_dotenv()

groq_key = os.getenv("GROQ_API_KEY")
google_key = os.getenv("GOOGLE_API_KEY")

print("Groq key loaded:", "✅" if groq_key else "❌ MISSING")
print("Google key loaded:", "✅" if google_key else "❌ MISSING")

Groq key loaded: ✅
Google key loaded: ✅


c:\Users\user\OneDrive\Capstone Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\user\AppData\Local\Temp\ipykernel_23632\3241140904.py:2: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


In [2]:
from google import genai
from dotenv import load_dotenv
import os

load_dotenv()

# Configure Gemini
client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))

# Test with a fake transcript
question = "Explain what a fraction is"
transcript = "A fraction is a part of a whole number. Like one half means you split something into two parts and take one."

prompt = f"""You are evaluating a middle school student's math explanation.

The teacher asked: {question}

The student explained verbally: {transcript}

Respond in this EXACT format:
SCORE: [1, 2, 3, or 4]
WHAT_WAS_RIGHT: [one encouraging sentence]
WHAT_TO_IMPROVE: [one specific suggestion]
TEACHER_NOTE: [one sentence for the teacher]
LANGUAGE_DETECTED: [English / Roman Urdu / Sindhi / Mixed]"""

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt
)

print(response.text)

SCORE: 3
WHAT_WAS_RIGHT: You correctly identified that a fraction represents a "part of a whole" and provided a clear, helpful example with one-half.
WHAT_TO_IMPROVE: To be more precise, explain that a fraction is a part of a "whole quantity" or "whole item" rather than specifically a "whole number."
TEACHER_NOTE: This explanation shows a solid grasp of the fundamental concept of fractions through a good definition and illustrative example.
LANGUAGE_DETECTED: English


In [3]:
from sqlalchemy import create_engine, Column, Integer, String, Text, DateTime
from sqlalchemy.ext.declarative import declarative_base
from sqlalchemy.orm import sessionmaker
from datetime import datetime

# Create database engine - this creates voicecheck.db file
engine = create_engine("sqlite:///voicecheck.db", echo=True)

Base = declarative_base()

# Sessions table - one row per teacher question
class Session(Base):
    __tablename__ = "sessions"
    
    id = Column(Integer, primary_key=True)
    question = Column(Text, nullable=False)
    student_link = Column(String, unique=True)
    created_at = Column(DateTime, default=datetime.utcnow)

# Responses table - one row per student submission
class Response(Base):
    __tablename__ = "responses"
    
    id = Column(Integer, primary_key=True)
    session_id = Column(Integer, nullable=False)
    audio_filename = Column(String)
    canvas_image_filename = Column(String)
    uploaded_image_filename = Column(String, nullable=True)
    transcript = Column(Text)
    score = Column(Integer)
    what_was_right = Column(Text)
    what_to_improve = Column(Text)
    teacher_note = Column(Text)
    language_detected = Column(String)
    submitted_at = Column(DateTime, default=datetime.utcnow)

# Create all tables
Base.metadata.create_all(engine)
print("✅ Database and tables created successfully!")

2026-05-02 11:44:15,572 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-05-02 11:44:15,574 INFO sqlalchemy.engine.Engine PRAGMA main.table_info("sessions")
2026-05-02 11:44:15,576 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-05-02 11:44:15,578 INFO sqlalchemy.engine.Engine PRAGMA main.table_info("responses")
2026-05-02 11:44:15,581 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-05-02 11:44:15,583 INFO sqlalchemy.engine.Engine COMMIT
✅ Database and tables created successfully!


C:\Users\user\AppData\Local\Temp\ipykernel_23632\811364874.py:9: MovedIn20Warning: The ``declarative_base()`` function is now available as sqlalchemy.orm.declarative_base(). (deprecated since: 2.0) (Background on SQLAlchemy 2.0 at: https://sqlalche.me/e/b8d9)
  Base = declarative_base()


In [4]:
from sqlalchemy.orm import sessionmaker
import uuid

# Create a session to talk to the database
SessionLocal = sessionmaker(bind=engine)
db = SessionLocal()

# Insert a fake teacher session
fake_session = Session(
    question="Explain what a fraction is",
    student_link=str(uuid.uuid4()),
    created_at=datetime.utcnow()
)

db.add(fake_session)
db.commit()
db.refresh(fake_session)

print("✅ Session inserted!")
print("Session ID:", fake_session.id)
print("Question:", fake_session.question)
print("Student Link:", fake_session.student_link)
print("Created At:", fake_session.created_at)

2026-05-02 11:44:23,299 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-05-02 11:44:23,305 INFO sqlalchemy.engine.Engine INSERT INTO sessions (question, student_link, created_at) VALUES (?, ?, ?)
2026-05-02 11:44:23,306 INFO sqlalchemy.engine.Engine [generated in 0.00184s] ('Explain what a fraction is', '7e77fd1f-e5b0-4c88-8148-0378e5a1650d', '2026-05-02 06:44:23.294267')
2026-05-02 11:44:23,315 INFO sqlalchemy.engine.Engine COMMIT
2026-05-02 11:44:23,331 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-05-02 11:44:23,344 INFO sqlalchemy.engine.Engine SELECT sessions.id, sessions.question, sessions.student_link, sessions.created_at 
FROM sessions 
WHERE sessions.id = ?
2026-05-02 11:44:23,347 INFO sqlalchemy.engine.Engine [generated in 0.00268s] (3,)
✅ Session inserted!
Session ID: 3
Question: Explain what a fraction is
Student Link: 7e77fd1f-e5b0-4c88-8148-0378e5a1650d
Created At: 2026-05-02 06:44:23.294267


C:\Users\user\AppData\Local\Temp\ipykernel_23632\2106171592.py:12: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  created_at=datetime.utcnow()


In [5]:
from sqlalchemy import create_engine, text
from dotenv import load_dotenv
import os

load_dotenv()

USER = os.getenv("user")
PASSWORD = os.getenv("password")
HOST = os.getenv("host")
PORT = os.getenv("port")
DBNAME = os.getenv("dbname")

DATABASE_URL = f"postgresql+psycopg2://{USER}:{PASSWORD}@{HOST}:{PORT}/{DBNAME}?sslmode=require"

engine = create_engine(DATABASE_URL)

try:
    with engine.connect() as connection:
        result = connection.execute(text("SELECT version()"))
        print("✅ Connected to Supabase!")
        print(result.fetchone()[0])
except Exception as e:
    print(f"❌ Failed: {e}")

❌ Failed: (psycopg2.OperationalError) could not translate host name "db.dmqruveoffolfkhsyqjg.supabase.co" to address: Name or service not known

(Background on this error at: https://sqlalche.me/e/20/e3q8)


In [6]:
from dotenv import load_dotenv
import os

load_dotenv()

print("user:", os.getenv("user"))
print("host:", os.getenv("host"))
print("port:", os.getenv("port"))
print("dbname:", os.getenv("dbname"))
print("password:", "✅ set" if os.getenv("password") else "❌ missing")

user: postgres.abcdefghijk
host: db.dmqruveoffolfkhsyqjg.supabase.co
port: 5432
dbname: postgres
password: ✅ set


In [8]:
from dotenv import load_dotenv
import os

load_dotenv(override=True)  # ← forces reload of .env file

print("user:", os.getenv("user"))
print("host:", os.getenv("host"))
print("port:", os.getenv("port"))
print("dbname:", os.getenv("dbname"))
print("password:", "✅ set" if os.getenv("password") else "❌ missing")

user: postgres.abcdefghijk
host: db.dmqruveoffolfkhsyqjg.supabase.co
port: 5432
dbname: postgres
password: ✅ set


In [9]:
import groq
import os
from dotenv import load_dotenv

load_dotenv(override=True)

client = groq.Groq(api_key=os.getenv("GROQ_API_KEY"))

audio_path = "uploads/audio.ogg"

with open(audio_path, "rb") as audio_file:
    transcription = client.audio.transcriptions.create(
        model="whisper-large-v3",
        file=audio_file,
        response_format="text"
    )

print("✅ Transcript:")
print(transcription)

✅ Transcript:
 There are three angles in the triangle which add up to 180. And the ratios are 2, 3, and 5, which add up to 10. And then 10 equals to 180. One unit would be 180 upon 10, which equals 18. And then we have to do 18 times 2, 3, and 5. 2 times 18 is 36 which is the size of the smallest angle.


In [10]:
pip install pymupdf

   ---------------------------------------- 0.0/19.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/19.2 MB ? eta -:--:--
    --------------------------------------- 0.3/19.2 MB ? eta -:--:--
    --------------------------------------- 0.3/19.2 MB ? eta -:--:--
   - -------------------------------------- 0.5/19.2 MB 829.7 kB/s eta 0:00:23
   - -------------------------------------- 0.5/19.2 MB 829.7 kB/s eta 0:00:23
   - -------------------------------------- 0.5/19.2 MB 829.7 kB/s eta 0:00:23
   - -------------------------------------- 0.5/19.2 MB 829.7 kB/s eta 0:00:23
   - -------------------------------------- 0.5/19.2 MB 829.7 kB/s eta 0:00:23
   - -------------------------------------- 0.8/19.2 MB 387.3 kB/s eta 0:00:48
   - -------------------------------------- 0.8/19.2 MB 387.3 kB/s eta 0:00:48
   -- ------------------------------------- 1.0/19.2 MB 454.6 kB/s eta 0:00:41
   -- ------------------------------------- 1.3/19.2 MB 498.8 kB/s eta 0:00:36
   -- ---


[notice] A new release of pip is available: 25.2 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [11]:
import fitz  # pymupdf
import os

# Convert first page of PDF to image
pdf_path = "uploads/student1.pdf"
output_path = "uploads/working_image.png"

doc = fitz.open(pdf_path)
page = doc[0]  # first page only
mat = fitz.Matrix(2, 2)  # 2x zoom for better quality
pix = page.get_pixmap(matrix=mat)
pix.save(output_path)

print("✅ PDF converted to image successfully!")
print(f"Saved to: {output_path}")

✅ PDF converted to image successfully!
Saved to: uploads/working_image.png


In [12]:
from google import genai
from dotenv import load_dotenv
import os
import base64

load_dotenv(override=True)

client_gemini = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))

# Load and encode image
with open("uploads/working_image.png", "rb") as f:
    image_data = base64.b64encode(f.read()).decode("utf-8")

question = "A triangle has angles in the ratio 2:3:5. Find all three angles."

prompt = f"""You are evaluating a middle school student's math explanation.

The teacher asked: {question}

The student explained verbally: {transcription}

The image above shows the student's written working.
Analyse BOTH the verbal explanation AND the written working together.
If you spot errors in the written working that the student did not 
mention verbally — reference them specifically in feedback.

Respond in this EXACT format:
SCORE: [1, 2, 3, or 4]
WHAT_WAS_RIGHT: [one encouraging sentence referencing their actual words or working]
WHAT_TO_IMPROVE: [one specific suggestion referencing written working if error found]
TEACHER_NOTE: [one sentence for the teacher about what to follow up]
LANGUAGE_DETECTED: [English / Roman Urdu / Sindhi / Mixed]"""

response = client_gemini.models.generate_content(
    model="gemini-2.5-flash",
    contents=[
        {
            "parts": [
                {
                    "inline_data": {
                        "mime_type": "image/png",
                        "data": image_data
                    }
                },
                {
                    "text": prompt
                }
            ]
        }
    ]
)

print(response.text)

SCORE: 4
WHAT_WAS_RIGHT: The student demonstrated a clear and logical approach to solving the problem, correctly calculating the value of one ratio unit and then using it to find all three angles.
WHAT_TO_IMPROVE: To make the verbal explanation even clearer, the student could explicitly state "10 parts correspond to 180 degrees" rather than "10 equals to 180."
TEACHER_NOTE: The student has a strong grasp of ratio concepts and can apply them effectively; encourage them to always state all findings when the question asks for multiple values.
LANGUAGE_DETECTED: English


## Model Comparison Test
### Gemini 2.5 Flash vs Groq Llama 4 Scout
Testing which model gives better multimodal feedback for student explanations.
We will evaluate both on the same transcript and image and compare results.

In [1]:
# ── SHARED INPUTS ────────────────────────────────────────────────────────
import base64, os, groq
from google import genai
from dotenv import load_dotenv

load_dotenv(override=True)

question = "A triangle has angles in the ratio 2:3:5. Find all three angles."

transcript = """There are three angles in the triangle which add up to 180. 
And the ratios are 2, 3, and 5, which add up to 10. And then 10 equals to 180. 
One unit would be 180 upon 10, which equals 18. And then we have to do 18 times 
2, 3, and 5. 2 times 18 is 36 which is the size of the smallest angle."""

# Load image
with open("uploads/working_image.png", "rb") as f:
    image_data = base64.b64encode(f.read()).decode("utf-8")

prompt = f"""You are evaluating a middle school student's math explanation.

The teacher asked: {question}

The student explained verbally: {transcript}

The image shows the student's written working.
Analyse BOTH the verbal explanation AND the written working together.
If you spot errors in the written working not mentioned verbally, reference them.

Respond in this EXACT format:
SCORE: [1, 2, 3, or 4]
WHAT_WAS_RIGHT: [one encouraging sentence referencing their actual words or working]
WHAT_TO_IMPROVE: [one specific suggestion referencing written working if error found]
TEACHER_NOTE: [one sentence for the teacher about what to follow up]
LANGUAGE_DETECTED: [English / Roman Urdu / Sindhi / Mixed]"""

# ── MODEL 1: GEMINI 2.5 FLASH ─────────────────────────────────────────────
print("=" * 60)
print("MODEL 1: Gemini 2.5 Flash")
print("=" * 60)

client_gemini = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))

gemini_response = client_gemini.models.generate_content(
    model="gemini-2.5-flash",
    contents=[
        {
            "parts": [
                {
                    "inline_data": {
                        "mime_type": "image/png",
                        "data": image_data
                    }
                },
                {
                    "text": prompt
                }
            ]
        }
    ]
)

print(gemini_response.text)

# ── MODEL 2: GROQ LLAMA 4 SCOUT ───────────────────────────────────────────
print("\n" + "=" * 60)
print("MODEL 2: Groq Llama 4 Scout")
print("=" * 60)

client_groq = groq.Groq(api_key=os.getenv("GROQ_API_KEY"))

groq_response = client_groq.chat.completions.create(
    model="meta-llama/llama-4-scout-17b-16e-instruct",
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "type": "image_url",
                    "image_url": {
                        "url": f"data:image/png;base64,{image_data}"
                    }
                },
                {
                    "type": "text",
                    "text": prompt
                }
            ]
        }
    ]
)

print(groq_response.choices[0].message.content)

MODEL 1: Gemini 2.5 Flash
SCORE: 4
WHAT_WAS_RIGHT: The student demonstrated a complete understanding of how to use ratios to divide a quantity, correctly calculating all three angles of the triangle and explaining their process clearly.
WHAT_TO_IMPROVE: When verbally stating "10 equals to 180", it would be more precise to say "the total ratio units (10) correspond to 180 degrees"; additionally, the extraneous boxed numbers at the bottom of the written work should be removed for clarity.
TEACHER_NOTE: The student clearly understands the concept of ratios in geometry; ensure they consistently articulate the relationship between ratio sums and total quantities.
LANGUAGE_DETECTED: English

MODEL 2: Groq Llama 4 Scout
SCORE: 3
WHAT_WAS_RIGHT: The student correctly identified that the sum of the angles in a triangle is 180 degrees and used the ratio to find the size of the smallest angle.
WHAT_TO_IMPROVE: The student wrote 10 equals to 180 in both verbal and written explanation, which seems 

## Model Comparison Results

| Criteria | Gemini 2.5 Flash | Groq Llama 4 Scout |
|---|---|---|
| Score | 4 | 3 |
| Image analysis | Strong — spotted errors in written work | Weak — ignored image |
| Feedback quality | Highly specific and encouraging | Good but generic |
| Multimodal capability | Excellent | Limited |

### Decision
**Gemini 2.5 Flash selected for student feedback** because it genuinely 
analyses both voice transcript and written working together — the core 
feature of Explainly.

**Groq Llama 4 Scout used for AI question analysis** — text only task 
where speed matters more than vision capability.

### Architecture Decision
- Audio transcription → Groq Whisper (fastest)
- Question analysis → Groq Llama 4 Scout (fast, text only)
- Student feedback → Gemini 2.5 Flash (best multimodal)